# gramlab — map stage v2 (CPU): the meaning map trained for retrieval

Inputs: **gramlab** (Version 3: BM25 index, raw Wikipedia text, teacher files, scoreboard results) and optionally **gramlab-map**
(v1 results). **Accelerator: None**, Internet on, *Save & Run All*; ≈ 2 h (teacher passage embeddings on CPU + POS tagging).

v1 regressed single sentences onto the teacher and was useless as a retriever (recall@1 0.06). v2 keeps the idea — a 32-dim code
per word, summed per grammar slot — and changes the training signal: mean pooling, passages as inputs, question/body-sentence →
gold-passage pairs, teacher *passage* embeddings as targets. Both variants (7 slots vs bag of words), scored on the same scoreboard
as BM25 (recall@1 0.607 all / 0.233 body) and the teacher (0.590 / 0.331); the better variant's passage matrix is saved for chat.

In [ ]:
import os, subprocess, sys, time, glob, shutil, json
T0 = time.time()
def sh(cmd, must=True):
    print('\n$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if must and r.returncode != 0:
        raise SystemExit(f'command failed ({r.returncode}): {cmd}')
os.chdir('/kaggle/working')
if not os.path.exists('gramlab'):
    sh('git clone --depth 1 https://github.com/furevych/gramlab.git')
os.chdir('/kaggle/working/gramlab')
src = sorted(glob.glob('/kaggle/input/**/work/bm25/weights.npz', recursive=True))
if not src:
    raise SystemExit('Attach the gramlab notebook output (Version 3) as an input')
root = src[0].split('/work/')[0]; print('gramlab output:', root)
os.makedirs('work', exist_ok=True); os.makedirs('results', exist_ok=True); os.makedirs('data', exist_ok=True)
if not os.path.exists('work/bm25'):
    shutil.copytree(f'{root}/work/bm25', 'work/bm25')
for f in glob.glob(f'{root}/results/*.json') + glob.glob('/kaggle/input/**/results/map.json', recursive=True):
    shutil.copy(f, 'results/')
shutil.copy(f'{root}/data/stsb.jsonl', 'data/')
WIKI = ' '.join(f'--wiki {root}/data/{f}' for f in ['simplewiki.jsonl', 'enwiki.jsonl', 'enwiki2.jsonl'] if os.path.exists(f'{root}/data/{f}'))
try:
    import spacy; spacy.load('en_core_web_sm')
except Exception:
    sh('pip install -q spacy && python -m spacy download -q en_core_web_sm', must=False)
print(f'elapsed {(time.time()-T0)/60:.0f} min')

In [ ]:
sh(f'python scripts/map_stage2.py --index work/bm25 {WIKI} --queries work/bm25/eval_queries.jsonl --stsb data/stsb.jsonl '
   '--out work/map2 --results results/map2.json --train_passages 150000 --train_articles 150000 --epochs 6 --n_process 4')
print(f'elapsed {(time.time()-T0)/60:.0f} min')

## What the codes know: nearest words (v2, best variant)

In [ ]:
import numpy as np
sys.path.insert(0, '/kaggle/working/gramlab')
from gramlab.map import MapEncoder, SLOTS
best = json.load(open('results/map2.json'))['best_for_hybrid']
enc = MapEncoder(f'work/map2/{best}')
codes = enc.model.code.weight.detach().numpy()[:len(enc.vocab.words)]
codes_n = codes / (np.linalg.norm(codes, axis=1, keepdims=True) + 1e-9)
def nearest(word, k=8):
    i = enc.vocab.index.get(word)
    if i is None: return []
    s = codes_n @ codes_n[i]; order = np.argsort(-s)[1:k+1]
    return [(enc.vocab.words[j], round(float(s[j]), 2)) for j in order]
print('variant:', best)
for w in ['city', 'river', 'king', 'kill', 'hate', 'love', 'quickly', 'france', 'capital', 'born', 'strike', 'rage']:
    print(f'{w:10}', nearest(w))

In [ ]:
shutil.rmtree('/kaggle/working/gramlab/.git', ignore_errors=True)
os.system('du -sh /kaggle/working/gramlab/work/* /kaggle/working/gramlab/results 2>/dev/null')
print(f'total {(time.time()-T0)/60:.0f} min')